# EA Page Viewer

Simple notebook to view EA document pages for manual review and exploration.

In [1]:
import pandas as pd
from pathlib import Path

# Display settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

## Load Data

In [2]:
# Load CE documents
documents = pd.read_parquet('../data/analysis/documents_combined.parquet')
ea_documents = documents[documents["dataset_source"] == "EA"]
print(f"Loaded {len(ea_documents):,} EA documents")

# Load CE pages
ea_pages = pd.read_parquet('../data/processed/ea/pages.parquet')
print(f"Loaded {len(ea_pages):,} EA pages")

Loaded 14,242 EA documents
Loaded 469,106 EA pages


## Helper Functions

In [3]:
def list_documents(project_id):
    """List all documents for a given project."""
    docs = ea_documents[ea_documents["project_id"] == project_id]
    if len(docs) == 0:
        print(f"No documents found for project_id: {project_id}")
        return None
    
    print(f"\nDocuments for project: {project_id}")
    print("-" * 80)
    display_cols = ["document_id", "document_type", "document_type_category", 
                    "main_document", "file_name", "total_pages"]
    available_cols = [c for c in display_cols if c in docs.columns]
    return docs[available_cols]

In [4]:
def view_page(document_id, page_num=1):
    """View a specific page from a document."""
    page = ea_pages[(ea_pages["document_id"] == document_id) & 
                    (ea_pages["page_number"] == page_num)]
    
    if len(page) == 0:
        print(f"Page {page_num} not found for document_id: {document_id}")
        return None
    
    # Get document metadata
    doc = ea_documents[ea_documents["document_id"] == document_id]
    if len(doc) > 0:
        print(f"\n{'='*80}")
        print(f"Document: {doc.iloc[0].get('file_name', 'N/A')}")
        print(f"Type: {doc.iloc[0].get('document_type', 'N/A')} | "
              f"Category: {doc.iloc[0].get('document_type_category', 'N/A')} | "
              f"Main: {doc.iloc[0].get('main_document', 'N/A')}")
        print(f"{'='*80}")
    
    print(f"\n--- PAGE {page_num} ---\n")
    print(page.iloc[0]["page_text"])
    print(f"\n--- END PAGE {page_num} ---\n")
    return page.iloc[0]["page_text"]

In [5]:
def view_pages(document_id, start_page=1, end_page=5):
    """View a range of pages from a document."""
    # Get document metadata
    doc = ea_documents[ea_documents["document_id"] == document_id]
    if len(doc) > 0:
        print(f"\n{'='*80}")
        print(f"Document: {doc.iloc[0].get('file_name', 'N/A')}")
        print(f"Type: {doc.iloc[0].get('document_type', 'N/A')} | "
              f"Category: {doc.iloc[0].get('document_type_category', 'N/A')} | "
              f"Main: {doc.iloc[0].get('main_document', 'N/A')}")
        print(f"{'='*80}")
    
    for page_num in range(start_page, end_page + 1):
        page = ea_pages[(ea_pages["document_id"] == document_id) & 
                        (ea_pages["page_number"] == page_num)]
        if len(page) > 0:
            print(f"\n--- PAGE {page_num} ---\n")
            print(page.iloc[0]["page_text"])
            print(f"\n--- END PAGE {page_num} ---\n")
        else:
            print(f"Page {page_num} not found")
            break

In [6]:
def view_project_pages(project_id, document_id=None):
    """View all pages of each document in a project (no truncation).
    
    Args:
        project_id: The project ID to filter documents
        document_id: Optional document ID to view only that specific document
    """
    docs = ea_documents[ea_documents["project_id"] == project_id]
    
    # Filter by document_id if provided
    if document_id is not None:
        docs = docs[docs["document_id"] == document_id]
        
    if len(docs) == 0:
        if document_id is not None:
            print(f"No document found with document_id: {document_id} in project_id: {project_id}")
        else:
            print(f"No documents found for project_id: {project_id}")
        return
        
    for _, doc in docs.iterrows():
        doc_id = doc["document_id"]
        print(f"\n{'='*80}")
        print(f"Document: {doc.get('file_name', 'N/A')}")
        print(
            f"Type: {doc.get('document_type', 'N/A')} | "
            f"Category: {doc.get('document_type_category', 'N/A')} | "
            f"Main: {doc.get('main_document', 'N/A')}"
        )
        print(f"{'='*80}")
                
        doc_pages = (
            ea_pages[ea_pages["document_id"] == doc_id]
            .sort_values("page_number")
        )
                
        for _, page in doc_pages.iterrows():
            print(f"\n--- PAGE {page['page_number']} ---\n")
            print(page["page_text"])

In [7]:
def search_pages(project_id, search_term):
    """Search for a term in all pages of a project."""
    docs = ea_documents[ea_documents["project_id"] == project_id]
    doc_ids = docs["document_id"].tolist()
    
    matching_pages = ea_pages[
        (ea_pages["document_id"].isin(doc_ids)) & 
        (ea_pages["page_text"].str.contains(search_term, case=False, na=False))
    ]
    
    print(f"\nFound {len(matching_pages)} pages containing '{search_term}'")
    print("-" * 80)
    
    for _, page in matching_pages.iterrows():
        doc = docs[docs["document_id"] == page["document_id"]].iloc[0]
        print(f"\nDocument: {doc.get('file_name', 'N/A')} | Page: {page['page_number']}")
        
        # Show context around the search term
        text = page["page_text"]
        idx = text.lower().find(search_term.lower())
        if idx >= 0:
            start = max(0, idx - 200)
            end = min(len(text), idx + len(search_term) + 200)
            context = text[start:end]
            if start > 0:
                context = "..." + context
            if end < len(text):
                context = context + "..."
            print(context)
    
    return matching_pages

In [8]:
def get_random_project(n_docs=2):
    """Get a random project with at least n_docs documents."""
    import random
    doc_counts = ea_documents.groupby("project_id").size()
    eligible = doc_counts[doc_counts >= n_docs].index.tolist()
    
    project_id = random.choice(eligible)
    print(f"Random project with {doc_counts[project_id]} documents: {project_id}")
    return project_id

## Explore a specific project

Set the `project_id` below and run the cells to explore.

In [18]:
# Set your project ID here
project_id = "d27822aed3a7cb74d351d8b61572c584" # 
#project_id = "f5aa4d8a-9bf8-c3e0-2d08-a37ebda57ee9" # 

In [19]:
# List all documents for this project
list_documents(project_id)


Documents for project: d27822aed3a7cb74d351d8b61572c584
--------------------------------------------------------------------------------


,document_id,document_type,document_type_category,main_document,file_name,total_pages
117,d65f66dd15d2de803f276fbea35b3d91,,other,NO,WRV Recreation and Access Decision Record.pdf,8
118,8e7467583a541c65d9e4780c1d4ae11c,,other,NO,WRV_Recreation_and_Access_EA_Interested_Parties_Letter.pdf,2
119,fca2bb43e021c5df7c545d3eeb98c5a1,,other,NO,Map_10_Slaughterhouse_Proposed_Campsites_and_Trailhead_Alternatives_B_&_D.pdf,1
120,8d3f5cc86694b83fbf305fa2a95d10d8,,draft,NO,WRV Recreation and Access Draft EA_Public Comments and Responses.pdf,126
121,affea5a3c9ba176ff8fcc77ec9b02c10,,other,NO,Map_13_Seasonal_Restrictions_Alternative_B.pdf,1
122,1dbfc418c09f3aa34c2fd19585ee7e27,,other,NO,DR Map Seasonal Restrictions.pdf,1
123,f102c790589827195a475b33e33c9836,,other,NO,Map 15 Season Restrictions D.pdf,1
124,3af47f1b79df1deb13a8995f3310acb8,,other,YES,Specialist_Report_Issue_5_Water_Quality_Fisheries.pdf,18
125,62b233eb8fea552f308ff2691b9500b4,,other,NO,Map_3_Designation_Changes_Alternative_C.pdf,1
126,8ba42b2a1bbd60ceadc048eea8f4a34a,,other,NO,Map 19 LWC Alternative D.pdf,1


In [20]:
# View first 3 pages of each document
view_project_pages(project_id, document_id = "8d3f5cc86694b83fbf305fa2a95d10d8")
#view_project_pages(project_id)


Document: WRV Recreation and Access Draft EA_Public Comments and Responses.pdf
Type:  | Category: draft | Main: NO

--- PAGE 1 ---

1 
 
 
 
Wood River Valley Recreation and Access Environmental Assessment (EA) Response to Draft EA Comments.  Substantive Comment (BLM NEPA 
Handbook H-1790-1 definition) 1. Question, with reasonable basis, the accuracy of information in the EA, 2. question with reasonable basis, the 
adequacy of methodology for, or assumptions used for the environmental analysis, 3. present new information relevant to the analysis, 4. present 
reasonable alternatives other than those analyzed in the EA, 5. cause changes or revisions in one or more of the alternatives.  No response necessary for 
non-substantive comments.  
 
Number 
Public Comment  
BLM Response 


--- PAGE 10 ---

10 
 
damaged by heavy traffic and extremely prone to fire. Camping will 
increase traffic use throughout the area immensely and destroy the 
beauty of the canyon and increase the risk of fir

## View Specific Document

In [ ]:
# Get document IDs for the project
docs = ea_documents[ea_documents["project_id"] == project_id]
docs[["document_id", "file_name", "main_document", "document_type"]]

,document_id,file_name,main_document,document_type
73227,65c6bb9f-ea78-d49b-3513-f5fc9c09da57,DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf,NO,OTHER
73228,25334806-7f45-75fb-d4b0-cad8e1e5567a,DOI-BLM-CO-N010-2021-0043-CX_Road 1509 Emergency Repair_for web.pdf,YES,CE


In [14]:
# Set document_id and view specific pages
document_id = docs.iloc[0]["document_id"]  # First document
view_pages(document_id, start_page=1, end_page=5)


Document: DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf
Type: OTHER | Category: other | Main: NO
Page 1 not found


## Search for Text

In [32]:
# Search for dates or specific text
search_pages(project_id, "22")


Found 1 pages containing '22'
--------------------------------------------------------------------------------

Document: cx-007803.pdf | Page: 1-4
...of any spilled material shall
begin immediately.

10. All potential pitfalls to wildlife will be covered or filled when not attended.

Limetto King acting for
Linda Hughes
NEPA Compliance Officer

11/22/2011
Date

5


document_id page_number  \
73021  d0da2533-5bbd-70b4-796f-d8d0145303fe         1-4   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [16]:
# Search for decision-related text
search_pages(project_id, "approved")


Found 2 pages containing 'approved'
--------------------------------------------------------------------------------

Document: DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf | Page: 1-4
...e emergency repair of BLM Road 1509 in Moffat County, Colorado,
near the Little Snake/White River Field Office boundary. The repair is expected to consist of
layering compatible rock material and BLM-approved Geotech material, then installing a culvert
and armoring the inlet and outlet of the culvert with local rock material from a nearby rock pile,
within BLM standards. The road repairs are expected to b...

Document: DOI-BLM-CO-N010-2021-0043-CX_Road 1509 Emergency Repair_for web.pdf | Page: 1-11
...safety.
Conformance with the Land Use Plan
The Proposed Action is subject to and is in conformance (43 CFR 1610.5) with the following
land use plan:
Land Use Plan: Little Snake Record of Decision and Approved Resource Management Plan
(ROD/RMP)
Date Approved: October 2011
Decisi

document_id page_number  \
4688  65c6bb9f-ea78-d49b-3513-f5fc9c09da57         1-4   
4689  25334806-7f45-75fb-d4b0-cad8e1e5567a        1-11   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                